# Lab 16-02 · Poles, Zeros, Phase and Equalizers

Five parts, each built on `pz.py`. **Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import math
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from scipy import signal
import pz

## Part 1 · Walk the unit circle

In [2]:
b, a = [1, 1], [1, -0.5]
z, p, k = pz.zpk(b, a)
w = np.linspace(0, math.pi, 1001)
print("geometric == direct:", np.allclose(pz.freq_resp(b, a, w), pz.geometric(z, p, k, w)))
for wi in (0, math.pi / 2, math.pi):
    print(f"w = {wi:.4f}: |H| = {abs(pz.freq_resp(b, a, wi)[0]):.3f}")
fig, ax = plt.subplots(figsize=(4, 4))
t = np.linspace(0, 2 * math.pi, 400)
ax.plot(np.cos(t), np.sin(t), "k", lw=0.6)
ax.plot(z.real, z.imag, "o", mfc="none", ms=9, label="zero")
ax.plot(p.real, p.imag, "x", ms=9, label="pole")
for wi in (0, math.pi / 2, math.pi):
    e = np.exp(1j * wi)
    for c, col in ((z[0], "C0"), (p[0], "C1")):
        ax.annotate("", xy=(e.real, e.imag), xytext=(c.real, c.imag), arrowprops=dict(arrowstyle="->", color=col, lw=0.8))
ax.set_aspect("equal"); ax.legend(fontsize=8); ax.set_title("vectors to e^{jw} at w = 0, pi/2, pi", fontsize=9)
plt.show()

geometric == direct: True
w = 0.0000: |H| = 4.000
w = 1.5708: |H| = 1.265
w = 3.1416: |H| = 0.000


## Part 2 · Notch the hum

*Predict:* which notch is best 20 ms after the hum starts, and which after one second?

In [3]:
fs = 1000
n = np.arange(2000)
sig = np.cos(2 * math.pi * 10 * n / fs)
hum = np.where(n >= 500, 0.5 * np.cos(2 * math.pi * 50 * n / fs), 0.0)
x = sig + hum
w0 = 2 * math.pi * 50 / fs
fig, ax = plt.subplots(figsize=(8, 3))
for r in (0.9, 0.98, 0.995):
    bn, an = pz.notch(w0, r)
    resid = signal.lfilter(bn, an, hum)          # what the notch leaves of the hum alone
    ax.plot(n / fs, resid, lw=0.8, label=f"r = {r}")
    print(f"r = {r}: 10 Hz gain {abs(pz.freq_resp(bn, an, 2 * math.pi * 10 / fs)[0]):.4f}, "
          f"hum left 20 ms after onset {np.max(np.abs(resid[515:525])):.3f}, after 1 s {np.max(np.abs(resid[1490:1500])):.4f}, "
          f"time constant {-1 / math.log(r):.1f} samples")
ax.set_xlabel("t (s)"); ax.set_ylabel("residual hum"); ax.legend(fontsize=8)
plt.show()

r = 0.9: 10 Hz gain 0.9881, hum left 20 ms after onset 0.076, after 1 s 0.0000, time constant 9.5 samples
r = 0.98: 10 Hz gain 0.9995, hum left 20 ms after onset 0.328, after 1 s 0.0000, time constant 49.5 samples
r = 0.995: 10 Hz gain 1.0000, hum left 20 ms after onset 0.450, after 1 s 0.0035, time constant 199.5 samples


## Part 3 · Tune a resonator

In [4]:
from scipy.optimize import brentq
w0 = math.pi / 4
for r in (0.9, 0.95, 0.98, 0.99, 0.995, 0.999):
    b, a = pz.resonator(w0, r)
    W = np.linspace(0.3, 1.3, 200001)
    G = np.abs(pz.freq_resp(b, a, W)); k = int(np.argmax(G))
    f = lambda ww: abs(pz.freq_resp(b, a, ww)[0]) - G[k] / math.sqrt(2)
    bw = brentq(f, W[k], 1.3) - brentq(f, 0.3, W[k])
    print(f"r = {r}: Q = {W[k] / bw:7.2f}, rule {w0 / (2 * (1 - r)):7.2f}")

r = 0.9: Q =    3.78, rule    3.93
r = 0.95: Q =    7.68, rule    7.85


r = 0.98: Q =   19.45, rule   19.63
r = 0.99: Q =   39.08, rule   39.27
r = 0.995: Q =   78.35, rule   78.54
r = 0.999: Q =  392.50, rule  392.70


## Part 4 · Three phases, one magnitude

In [5]:
fam = {"minimum": [2, -2, 0.5], "linear": [1, -2.5, 1], "maximum": [0.5, -2, 2]}
w = np.linspace(0.001, math.pi, 500)
fig, (a1, a2, a3) = plt.subplots(1, 3, figsize=(11, 3))
for name, h in fam.items():
    a1.plot(w, np.abs(pz.freq_resp(h, [1], w)), label=name)
    a2.plot(range(3), pz.partial_energy(h) / pz.partial_energy(h)[-1], "o-", label=name)
    a3.plot(w, pz.group_delay(h, [1], w), label=name)
    print(f"{name:8s}: first-sample energy {pz.partial_energy(h)[0] / pz.partial_energy(h)[-1]:.4f}, DC group delay {pz.group_delay(h, [1], 0.0)[0]:.1f}")
a1.set_title("|H| (identical)", fontsize=9); a2.set_title("partial energy", fontsize=9); a3.set_title("group delay (samples)", fontsize=9)
a3.legend(fontsize=8)
plt.show()

minimum : first-sample energy 0.4848, DC group delay -2.0
linear  : first-sample energy 0.1212, DC group delay 1.0
maximum : first-sample energy 0.0303, DC group delay 4.0


## Part 5 · Equalize a link

*Predict:* does doubling $K$ from 10 to 21 halve the error rate at $\sigma$ = 0.3?

In [6]:
rng = np.random.default_rng(42)
N = 200_000
s = rng.choice([-1.0, 1.0], N)
noise = rng.normal(size=N)
for sigma in (0.1, 0.3):
    rx = np.convolve(s, [0.8, 1.0])[:N] + sigma * noise
    print(f"sigma = {sigma}: no equalizer, error rate {100 * np.mean(np.sign(rx[1:]) != s[:-1]):.2f}%")
    for K in (5, 10, 21):
        e = pz.anticausal_inverse(0.8, 1.0, K)
        y = np.convolve(rx, e)[:N]
        ber = np.mean(np.sign(y[K:]) != s[:N - K])
        print(f"   K = {K:2d}: eye opening {1 - 0.8**K:.3f}, noise gain {np.sqrt(np.sum(e**2)):.3f}, error rate {100 * ber:.3f}%")

sigma = 0.1: no equalizer, error rate 1.16%
   K =  5: eye opening 0.672, noise gain 1.575, error rate 0.001%
   K = 10: eye opening 0.893, noise gain 1.657, error rate 0.000%
   K = 21: eye opening 0.991, noise gain 1.667, error rate 0.000%
sigma = 0.3: no equalizer, error rate 12.66%
   K =  5: eye opening 0.672, noise gain 1.575, error rate 4.009%
   K = 10: eye opening 0.893, noise gain 1.657, error rate 2.491%
   K = 21: eye opening 0.991, noise gain 1.667, error rate 2.300%


**Discussion.** Answer the four questions in the [lab handout](README.md) before reading its answers.